In [1]:
import pandas as pd
import torch
import tqdm

from agfn.gym import MAG_HITL, LogReward
from agfn.policies import ForwardPolicyHITL, BackwardPolicyHITL

from agfn.gflownet import GFlowNet

from agfn.commons import is_mag

torch.manual_seed(42)

In [2]:
device = "cpu"
assert device == "cpu", "Currently, only CPU is supported"

## Load data

In [3]:
data_path = "data/chain4_dat01.csv"
data = pd.read_csv(data_path)
data = torch.tensor(data.values, dtype=torch.get_default_dtype(), device=device)

num_nodes = data.shape[1]

In [4]:
hidden_dim = 128
num_layers = 2

pf = ForwardPolicyHITL(num_nodes, hidden_dim, num_layers, device=device)
pb = BackwardPolicyHITL(device=device)

gfn = GFlowNet(pf, pb, criterion="tb", device=device)

In [5]:
log_reward = LogReward(data, device=device)

initial_state = MAG_HITL(num_nodes, 256, log_reward)
log_reward.warmup(gfn, initial_state=initial_state, num_batches=8)

In [6]:
state = gfn.sample(
    initial_state
)

p, s = state.parents_and_spouses

is_graph_mag, th = is_mag(p, s, return_th=True)
assert is_graph_mag.all(), (th.mean(), th)

## Train an AGFN

In [ ]:
opt = torch.optim.Adam([
    {"params": gfn.pf.parameters(), "lr": 1e-3},
    {"params": gfn.log_z, "lr": 1e-1},
])

num_epochs = int(1e4)

pbar = tqdm.tqdm(range(num_epochs))

for _ in pbar:
    initial_state = MAG_HITL(num_nodes, 256, log_reward)
    opt.zero_grad()
    loss = gfn(initial_state)
    loss.backward()
    opt.step()

    pbar.set_postfix(loss=loss.item())

  0%|          | 5/10000 [00:00<07:55, 21.00it/s, loss=42.1]

[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]


  0%|          | 8/10000 [00:00<08:32, 19.49it/s, loss=41.6]

[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]


  0%|          | 12/10000 [00:00<08:50, 18.82it/s, loss=42.7]

[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]


... (71 outputs left out)

[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]


  2%|▏         | 195/10000 [00:09<07:09, 22.83it/s, loss=42.3]

[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None, None, None]


  2%|▏         | 198/10000 [00:09<07:19, 22.29it/s, loss=41.8]